# Final cohort filtering

In [1]:
### that were custom made for this project
source('../00-utilities/functions/00_functions_data_loading.R')
file.sources <- list.files(
    path = "./ndmmFH1_helper_functions", 
    pattern = "\\.R$", full.names = TRUE
)
file.sources
# Source each R file
invisible(sapply(file.sources, source))

Loading required package: data.table

Loading required package: ggplot2

Loading required package: dplyr


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ lubridate 1.9.3     ✔ tibble    3.3.0
✔ purrr     1.1.0     ✔ tidyr     1.3.1
✔ readr     2.1.5     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::between()     masks data.table::between()
✖ dplyr::filter()      masks stats::filter()
✖ dplyr::first()       masks data.table::first()
✖ lubridate::hour()    masks data.table::hour()
✖ lubridate::isoweek() masks data.table::isoweek()
✖ dplyr::lag()         masks stats::lag()
✖ dplyr::last(

[1] "./ndmmFH1_helper_functions/data_crud.R" 
[2] "./ndmmFH1_helper_functions/data_manip.R"
[3] "./ndmmFH1_helper_functions/data_vis.R"  
[4] "./ndmmFH1_helper_functions/misc.R"      
[5] "./ndmmFH1_helper_functions/static.R"

# Load Plasma & Bone Marrow Olink Data 

In [2]:
### Load Plasma Data 
plasma_filename = "../../data/olink/Olink_Plasma_FH1_BR1_BR2_with_hise_descriptors.csv"
plasma <- fread(plasma_filename)
plasma = plasma[,-1, with=F]

### Get Bone Marrow data 
bm_filename = "../../data/olink/Olink_BM_with_hise_descriptors_08-18-2025.csv"
bm <- fread(bm_filename)

## Format Plasma Visits Data

In [4]:
sort(unique(plasma$visitDetails))

[1] "Immune Variation Day 0"          "Immune Variation Day 7"         
 [3] "Immune Variation Day 90"         "MM End Induction 1st Draw"      
 [5] "MM Post Induction 2-Cycles"      "MM Post Transplant 1 year"      
 [7] "MM Post Transplant 2 year"       "MM Post Transplant 60 Days"     
 [9] "MM Post Transplant 90 Days"      "MM Pre-Treatment"               
[11] "N/A - Flu-Series Timepoint Only" "N/A - stand-alone collection"   
[13] "Other"

In [5]:
plasma = plasma %>%
    dplyr::mutate(
      visitDetails = forcats::fct_recode(
        visitDetails,
          "PreTx" = "MM Pre-Treatment",
          "PI2C"= "MM Post Induction 2-Cycles",
          "EI" = "MM End Induction 1st Draw",
          "ASCT60d" ="MM Post Transplant 60 Days",
          "ASCT90d" = "MM Post Transplant 90 Days",
          "ASCT1y"="MM Post Transplant 1 year",
          "ASCT2y"= "MM Post Transplant 2 year", 
          "Imm d0" = "Immune Variation Day 0",
          "Imm d7" = "Immune Variation Day 7",
          "Imm d90" = "Immune Variation Day 90",
          Flu = "N/A - Flu-Series Timepoint Only", # flu day 0, 7, 90
          Standalone = "N/A - stand-alone collection", # sometimes flu day 0, sometimes before day 0
          Other = "Other",
      ),
      visitDetails = forcats::fct_relevel(
        visitDetails,
        "PreTx",
        "PI2C",
        "EI",
        "ASCT60d",
        "ASCT90d",
        "ASCT1y",
        "ASCT2y", 
        "Flu",
        "Standalone",
        "Other",
        "Imm d0",
        "Imm d7",
        "Imm d90"
      )
    ) 

In [6]:
table(plasma$visitDetails)


     PreTx       PI2C         EI    ASCT60d    ASCT90d     ASCT1y     ASCT2y 
     42279      40094      41581      28682      10684      30929      15480 
       Flu Standalone      Other     Imm d0     Imm d7    Imm d90 
   1166053       5524       7345     136881     136881     127959 

In [7]:
plasma = plasma %>%
    dplyr::mutate(Age = lubridate::year(drawDate) - birthYear) %>%
    dplyr::mutate(
      Subject = forcats::fct_reorder(
        Subject,
        as.numeric(substring(
          as.character(Subject),
          nchar(as.character(Subject)) -
            1,
          nchar(as.character(Subject))
        ))
      )
    )

## Format Bone Marrow Visits 

In [8]:
table(bm$VisitDetails)


   MM End Induction 1st Draw    MM Post Transplant 1 year 
                       47088                        32373 
   MM Post Transplant 2 year   MM Post Transplant 90 Days 
                       14715                        41202 
            MM Pre-Treatment N/A - stand-alone collection 
                       73575                         2943 

In [9]:
bm = bm %>%
    dplyr::mutate(
      visitDetails = forcats::fct_recode(
        VisitDetails,
          "PreTx" = "MM Pre-Treatment",
          "EI" = "MM End Induction 1st Draw",
          "ASCT90d" = "MM Post Transplant 90 Days",
          "ASCT1y"="MM Post Transplant 1 year",
          "ASCT2y"= "MM Post Transplant 2 year", 
          Standalone = "N/A - stand-alone collection", # sometimes flu day 0, sometimes before day 0
          Other = "",
      ),
      visitDetails = forcats::fct_relevel(
        visitDetails,
        "PreTx",
        "EI",
        "ASCT90d",
        "ASCT1y",
        "ASCT2y", 
        "Standalone",
        "Other"
      )
    ) 

table(bm$visitDetails)

Warning message:
“There were 2 warnings in `dplyr::mutate()`.
The first warning was:
ℹ In argument: `visitDetails = forcats::fct_recode(...)`.
Caused by warning:
! Unknown levels in `f`: 
ℹ Run `dplyr::last_dplyr_warnings()` to see the 1 remaining warning.”



     PreTx         EI    ASCT90d     ASCT1y     ASCT2y Standalone 
     73575      47088      41202      32373      14715       2943 

# Filter to just our selected subjects
- FH1 non-darataumamab treated subjects
  - excludes 'FH1020',
    'FH1022',
    'FH1023',
    'FH1024',
    'FH1026',
    'FH1027',
    'FH1028'
- BR1 and BR2 matched healthy subjects 

In [10]:
# Load our selected samples # TODO Load in from HISE
files <- hise::readFiles(fileIds = list("df721468-7adf-4a37-a86f-232b67392733"))
selected_samples <- read.csv(files[[1]]$file)

[1] "downloading fileID df721468-7adf-4a37-a86f-232b67392733"


In [12]:
plasma <- plasma %>% filter(Subject %in% selected_samples$Subject)
plasma = annotateFluResponse(plasma)
dim(plasma)

[1] 862142     45

In [14]:
bm <- bm %>% filter(Subject %in% selected_samples$Subject)
bm <- annotateFluResponse(bm)
dim(bm)

[1] 173637     27

In [15]:
unique(bm$Subject)
table(bm$manual.flu_response, useNA = 'always')

[1] "FH1018" "FH1008" "FH1011" "FH1007" "FH1014" "FH1017" "FH1004" "FH1016"
 [9] "FH1006" "FH1002" "FH1003" "FH1009" "FH1005" "FH1021" "FH1001" "FH1010"
[17] "FH1012"


non_responder     responder          <NA> 
        73575         82404         17658 

# Split Plasma to Relabel Flu Visits

## Plasma
As seen below, by splitting the visits into MM treatment visits ('PreTx', 'PI2C','EI','ASCT60d','ASCT90d','ASCT1y','ASCT2y'), 
and the visits according to their flu vaccine, this essentially splits the dataset cleanly with the exception of 2 samples that are labeled:

- ASCT90d and Flu Year Stand alone 1
- ASCT1y and Flu Year 1 Day 90

The rest of the treatment visits are labeled accurately, and can confidently be subset into an individual matrix. 

In [16]:
table(plasma[-grep('COVID-19', plasma$visitName)]$visitName, plasma[-grep('COVID-19', plasma$visitName)]$visitDetails)

                        
                         PreTx  PI2C    EI ASCT60d ASCT90d ASCT1y ASCT2y   Flu
  Flu Year 1 Day 0           0     0     0       0       0      0      0 66240
  Flu Year 1 Day 7           0     0     0       0       0      0      0 65876
  Flu Year 1 Day 90          0     0     0       0       0   1472      0 68426
  Flu Year 1 Stand-Alone     0     0     0       0    2944      0      0 32020
  Flu Year 2 Day 0           0     0     0       0       0      0      0 69838
  Flu Year 2 Day 7           0     0     0       0       0      0      0 67093
  Flu Year 2 Day 90          0     0     0       0       0   1472      0 70766
  Flu Year 2 Stand-Alone     0     0     0       0       0      0      0 33128
  Flu Year 3 Stand-Alone     0     0     0       0       0      0      0 26241
  Other - Non-Flu        24994 22065 25009   20578       0  26528  15480     0
                        
                         Standalone Other Imm d0 Imm d7 Imm d90
  Flu Year 1 Day 

In [17]:
# Treatment timepoint data split
plasmaVisitNames <- c('PreTx', 'PI2C','EI','ASCT60d','ASCT90d','ASCT1y','ASCT2y')

plasma_treatment <- plasma %>%
    filter(visitDetails %in% plasmaVisitNames)

# Set factors by visit
plasma_treatment$visitDetails = factor(
    plasma_treatment$visitDetails,
    levels = c('PreTx', 'PI2C','EI','ASCT60d','ASCT90d','ASCT1y','ASCT2y')
)

In [18]:
table(plasma$visitDetails, plasma$Cohort)

            
                BR1    BR2    FH1
  PreTx           0      0  24994
  PI2C            0      0  22065
  EI              0      0  25009
  ASCT60d         0      0  20578
  ASCT90d         0      0   2944
  ASCT1y          0      0  30929
  ASCT2y          0      0  15480
  Flu         10304 315610 239684
  Standalone      0      0   2944
  Other           0      0   7345
  Imm d0       1472  47104      0
  Imm d7       1472  47104      0
  Imm d90      1472  45632      0

To extract the flu vaccine data, we will keep Flu years 1-2, d0,d7,d90. 

In [19]:
# Flu data timepoint split
fluVisitNames <- c(
  'Flu Year 1 Day 0',
  'Flu Year 1 Day 7',
  'Flu Year 1 Day 90',
  'Flu Year 2 Day 0',
  'Flu Year 2 Day 7',
  'Flu Year 2 Day 90'
)

# Filter to these visitNames
# Here we do not filter to just FH1 cohort
# BR1 and BR2 cohorts have the same fluVisitNames :))
plasma_flu <- plasma %>%
    filter(visitName %in% fluVisitNames)

# Rename standalone to Day 0
# Justification: standalone collection is all 
# on day 0 or prior
plasma_flu[visitName == 'Flu Year 1 Stand-Alone', visitName := "Flu Year 1 Day 0"]
plasma_flu[visitName == 'Flu Year 2 Stand-Alone', visitName := "Flu Year 2 Day 0"]

# For downstream compatibility, overwrite the 
# visitDetails column to the ordered visitName for flu
# since flu timepoint details were only described in visitName
plasma_flu$visitDetails_old <- plasma_flu$visitDetails
plasma_flu$visitDetails <- factor(plasma_flu$visitName, levels = fluVisitNames)

plasma_flu = plasma_flu %>%
    dplyr::mutate(
      visitDetails = forcats::fct_recode(
        visitDetails,
          'Flu_Y1D0' = 'Flu Year 1 Day 0',
          'Flu_Y1D7' = 'Flu Year 1 Day 7',
          'Flu_Y1D90'= 'Flu Year 1 Day 90',
          'Flu_Y2D0' = 'Flu Year 2 Day 0',
          'Flu_Y2D7' = 'Flu Year 2 Day 7',
          'Flu_Y2D90'= 'Flu Year 2 Day 90'
      ),
      visitDetails = forcats::fct_relevel(
        visitDetails,
          'Flu_Y1D0', 
          'Flu_Y1D7', 
          'Flu_Y1D90',
          'Flu_Y2D0',
          'Flu_Y2D7',
          'Flu_Y2D90'
      )
    ) 
table(plasma_flu$visitDetails)


 Flu_Y1D0  Flu_Y1D7 Flu_Y1D90  Flu_Y2D0  Flu_Y2D7 Flu_Y2D90 
    67712     67348     71355     69838     67093     72238 

In [20]:
dim(plasma_flu)
dim(plasma_treatment)

[1] 415584     46

[1] 141999     45

In [21]:
# Recombine flu and treatment timepoints
plasma_combined <- rbind(plasma_flu, plasma_treatment, fill=TRUE)

# Annotate flu response
plasma_combined <- annotateFluResponse(plasma_combined)

In [22]:
table(plasma_combined$manual.flu_response, useNA = 'always')
table(plasma_combined$visitDetails)


non_responder     responder          <NA> 
       114424        147173        295986 


 Flu_Y1D0  Flu_Y1D7 Flu_Y1D90  Flu_Y2D0  Flu_Y2D7 Flu_Y2D90     PreTx      PI2C 
    67712     67348     71355     69838     67093     72238     24994     22065 
       EI   ASCT60d   ASCT90d    ASCT1y    ASCT2y 
    25009     20578      2944     30929     15480 

# BM Confirm Only Non-Ig Depleted Samples

In [23]:
# Confirm Removeal of IG- (IG depleted) samples
bm <- bm %>% dplyr::filter(grepl("-001-003", bm$SampleID))
print(table(bm$`Sample Type`=='Ig Non depleted', useNA = 'always')) # Must be TRUE


  TRUE   <NA> 
170694      0 


# Remap column names

In [24]:
colnames(bm)

[1] "SampleID"             "OlinkID"              "UniProt"             
 [4] "Assay"                "MissingFreq"          "Panel"               
 [7] "QC_Warning"           "LOD"                  "NPX"                 
[10] "Normalization"        "AssayWarning"         "Cohort"              
[13] "Subject"              "Sex"                  "VisitName"           
[16] "VisitDetails"         "DaysSinceFirstVisit"  "SampleKitGuid"       
[19] "is_technical_control" "SampleFormat"         "Sample Type"         
[22] "Age"                  "Race"                 "Visit"               
[25] "CollectionDate"       "visitDetails"         "manual.flu_response"

In [25]:
colnames(plasma_combined)

[1] "SampleID"                             
 [2] "Index"                                
 [3] "AssayUnique"                          
 [4] "OlinkID"                              
 [5] "UniProt"                              
 [6] "Assay"                                
 [7] "Panel"                                
 [8] "PlateID"                              
 [9] "Normalization"                        
[10] "Assay_Warning"                        
[11] "sampleKitGuid"                        
[12] "UniqueAssay"                          
[13] "N_Bridge"                             
[14] "BatchOffset"                          
[15] "NPX"                                  
[16] "LOD"                                  
[17] "sampleWarningBridge"                  
[18] "BridgingDetails"                      
[19] "filename"                             
[20] "BatchID"                              
[21] "projectGuid.projectGuid"              
[22] "lastUpdated.lastUpdated"              
[23] "labLastModified.labLastModified"      
[24] "surveyLastModified.surveyLastModified"
[25] "is_bridgingControl"                   
[26] "visitName"                            
[27] "visitDetails"                         
[28] "drawDate"                             
[29] "daysSinceFirstVisit"                  
[30] "file.fileType"                        
[31] "Sex"                                  
[32] "birthYear"                            
[33] "Ethnicity"                            
[34] "Race"                                 
[35] "Subject"                              
[36] "Cohort"                               
[37] "lab.id"                               
[38] "lab.sampleKitGuid"                    
[39] "lab.revisionNumber"                   
[40] "diseaseStatesRecordedAtVisit"         
[41] "project.short_name"                   
[42] "project.name"                         
[43] "sampleWarningBride"                   
[44] "Age"                                  
[45] "manual.flu_response"                  
[46] "visitDetails_old"

In [26]:
# Renaming columns as per `rename_cols`
plasma_rename_cols <- c(
  SampleID = "specimen.specimenGuid",
  OlinkID = "olink.assay_id",
  UniProt = "olink.uniprot_id",
  Assay = "olink.assay",
  Panel = "olink.panel",
  PlateID = "olink.plate_id",
  sampleKitGuid = "sample.sampleKitGuid",
  BatchOffset = "olink.norm_offset",
  NPX = "olink.NPX_norm",
  LOD = "olink.LOD_norm",
  visitName = "sample.visitName", 
  visitDetails = "sample.visitDetails",
  drawDate = "sample.drawDate",
  daysSinceFirstVisit = "sample.daysSinceFirstVisit",
  diseaseStatesRecordedAtVisit = "sample.diseaseStatesRecordedAtVisit",
  Sex = "subject.biologicalSex",
  birthYear = "subject.birthYear",
  Ethnicity = "subject.ethnicity",
  Race = "subject.race",
  Subject = "subject.subjectGuid",
  Cohort = "cohort.cohortGuid",
  Age = "subject.ageAtFirstDraw",
  manual.flu_response = "manual.flu_response"
)

bm_rename_cols <- c(
  SampleID = "specimen.specimenGuid",
  OlinkID = "olink.assay_id",
  UniProt = "olink.uniprot_id",
  Assay = "olink.assay",
  Panel = "olink.panel",
  #PlateID = "olink.plate_id",
  LOD = "olink.LOD_norm",
  NPX = "olink.NPX_norm",
  Subject = "subject.subjectGuid",
  Sex = "subject.biologicalSex",
  Age = "subject.ageAtFirstDraw",
  Race = "subject.race",
  Cohort = "cohort.cohortGuid",
  CollectionDate = "CollectionDate",
  SampleKitGuid = "SampleKitGuid",
  visitDetails = "sample.visitDetails",
  manual.flu_response = "manual.flu_response"
)

In [27]:
cbind(names(bm), bm_rename_cols)

Warning message in cbind(names(bm), bm_rename_cols):
“number of rows of result is not a multiple of vector length (arg 2)”


,bm_rename_cols
SampleID,specimen.specimenGuid
OlinkID,olink.assay_id
UniProt,olink.uniprot_id
Assay,olink.assay
MissingFreq,olink.panel
Panel,olink.LOD_norm
QC_Warning,olink.NPX_norm
LOD,subject.subjectGuid
NPX,subject.biologicalSex
Normalization,subject.ageAtFirstDraw


In [28]:
# Reverse the name and value in plasma_rename_cols
plasma_rename_cols_rev <- setNames(names(plasma_rename_cols), plasma_rename_cols)

# Reverse the name and value in bm_rename_cols
bm_rename_cols_rev <- setNames(names(bm_rename_cols), bm_rename_cols)

These columns will be removed from plasma:
```
  Index = "", # Remove
  AssayUnique = "", # Remove
  Normalization = "", # Remove
  Assay_Warning = "", # Remove
  UniqueAssay = "", # Remove
  N_Bridge = "", # Remove
  sampleWarningBridge = "", # Remove
  BridgingDetails = "", # Remove
  filename = "", # Remove
  BatchID = "", # Remove
  is_bridgingControl = "", # Remove
  file.fileType = "", # Remove
  sampleWarningBride = "", # Remove
  visitDetails_old = "", # Remove
```
and these will be removed from BM:
```
  Index = "",
  MissingFreq = "",
  Panel_Lot_Nr = "",
  QC_Warning = "",
  Normalization = "",
  Assay_Warning = "",
  is_technical_control = "",
  SampleType = "",
  Visit = "",
  is_normal_BM = "",
  is_MM_sample = "",
  AssayUnique = "",
```

In [29]:
all(plasma_rename_cols_rev %in% colnames(plasma_combined))

[1] TRUE

In [80]:
plasma_combined_renamed <- plasma_combined %>%
  select(all_of(plasma_rename_cols_rev)) %>%
  setNames(names(plasma_rename_cols_rev))
colnames(plasma_combined_renamed)

[1] "specimen.specimenGuid"               "olink.assay_id"                     
 [3] "olink.uniprot_id"                    "olink.assay"                        
 [5] "olink.panel"                         "olink.plate_id"                     
 [7] "sample.sampleKitGuid"                "olink.norm_offset"                  
 [9] "olink.NPX_norm"                      "olink.LOD_norm"                     
[11] "sample.visitName"                    "sample.visitDetails"                
[13] "sample.drawDate"                     "sample.daysSinceFirstVisit"         
[15] "sample.diseaseStatesRecordedAtVisit" "subject.biologicalSex"              
[17] "subject.birthYear"                   "subject.ethnicity"                  
[19] "subject.race"                        "subject.subjectGuid"                
[21] "cohort.cohortGuid"                   "subject.ageAtFirstDraw"             
[23] "manual.flu_response"

In [81]:
bm_renamed <- bm %>%
  select(all_of(bm_rename_cols_rev)) %>%
  setNames(names(bm_rename_cols_rev))
colnames(bm_renamed)

[1] "specimen.specimenGuid"  "olink.assay_id"         "olink.uniprot_id"      
 [4] "olink.assay"            "olink.panel"            "olink.LOD_norm"        
 [7] "olink.NPX_norm"         "subject.subjectGuid"    "subject.biologicalSex" 
[10] "subject.ageAtFirstDraw" "subject.race"           "cohort.cohortGuid"     
[13] "CollectionDate"         "SampleKitGuid"          "sample.visitDetails"   
[16] "manual.flu_response"

# Add CMV from subject metadata

In [82]:
fres <- hise::readFiles(list("a9d139fe-b944-454d-863a-9761b3d2118a"))

[1] "downloading fileID a9d139fe-b944-454d-863a-9761b3d2118a"


In [83]:
metadata <- fread(fres[[1]]$file)

In [84]:
head(metadata,2)

V1,subject.subjectGuid,cohort.cohortGuid,subject.biologicalSex,subject.race,subject.ethnicity,subject.birthYear,subject.ageAtEnrollment,sample.visitName,sample.visitDetails,⋯,RISS.stage,Notes,subject.bmi,subject.ageAtFirstDraw,subject.covidVaxDose1.daysSinceFirstVisit,subject.covidVaxDose2.daysSinceFirstVisit,sample.drawYear,sample.subjectAgeAtDraw,specimen.specimenGuid,pipeline.fileGuid
<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<chr>,<chr>,⋯,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<chr>,<chr>
0,FH1001,FH1,Female,Caucasian,Non-Hispanic origin,1956,64,Other - Non-Flu,MM Pre-Treatment,⋯,,,NA,NA,NA,NA,NA,NA,,
1,FH1002,FH1,Female,Caucasian,Non-Hispanic origin,1957,63,Other - Non-Flu,MM Pre-Treatment,⋯,II,,NA,NA,NA,NA,NA,NA,,


In [85]:
dim(plasma_combined_renamed)
dim(bm_renamed)

[1] 557583     23

[1] 170694     16

In [86]:
plasma_combined_renamed <- left_join(plasma_combined_renamed, unique(metadata[,c('subject.subjectGuid', 'subject.cmv')]))
dim(plasma_combined_renamed)

Joining with `by = join_by(subject.subjectGuid)`


[1] 557583     24

In [87]:
bm_renamed <- left_join(bm_renamed, unique(metadata[,c('subject.subjectGuid', 'subject.cmv')]))
dim(bm_renamed)

Joining with `by = join_by(subject.subjectGuid)`


[1] 170694     17

In [88]:
plasma_combined_renamed$sample.visitDetails <- factor(plasma_combined_renamed$sample.visitDetails,
                                                      levels=c('PreTx','PI2C','EI','ASCT60d',
                                                               'ASCT90d', 'ASCT1y','Flu_Y1D0',
                                                               'Flu_Y1D7', 'Flu_Y1D90', 'ASCT2y',
                                                               'Flu_Y2D0', 'Flu_Y2D7', 'Flu_Y2D90'))

bm_renamed$sample.visitDetails <- factor(bm_renamed$sample.visitDetails,
                                                      levels=c('PreTx','PI2C','EI','ASCT60d',
                                                               'ASCT90d', 'ASCT1y','Flu_Y1D0',
                                                               'Flu_Y1D7', 'Flu_Y1D90', 'ASCT2y',
                                                               'Flu_Y2D0', 'Flu_Y2D7', 'Flu_Y2D90'))
                                                               

# Remove Proteins that are All NA

In [89]:
dim(plasma_combined_renamed)
### Remove proteins that 
### do not have expression 
protein_missing_ness = plasma_combined_renamed[, mean(is.na(olink.NPX_norm)), by=olink.assay]
data.frame(table(protein_missing_ness$V1))
plasma_proteinsToKeep <- protein_missing_ness[V1!=1,]$olink.assay

plasma_combined_renamed <- plasma_combined_renamed[plasma_combined_renamed$olink.assay %in% plasma_proteinsToKeep,]
dim(plasma_combined_renamed)

[1] 557583     24

Var1,Freq
<fct>,<int>
0,1440
0.0209580838323353,1
0.0538922155688623,3
0.0658682634730539,2
0.0838323353293413,1
0.090032154340836,1
0.0989399293286219,4
0.107692307692308,11
1,1463


[1] 490170     24

In [90]:
dim(bm_renamed)
### Remove proteins that 
### do not have expression 
protein_missing_ness = bm_renamed[, mean(is.na(olink.NPX_norm)), by=olink.assay]
data.frame(table(protein_missing_ness$V1))
bm_proteinsToKeep <- protein_missing_ness[V1!=1,]$olink.assay

bm_renamed <- bm_renamed[bm_renamed$olink.assay %in% bm_proteinsToKeep,]
dim(bm_renamed)

[1] 170694     17

Var1,Freq
<fct>,<int>
0,2856
1,69


[1] 166692     17

# Save Files

In [91]:
file.name <- "MM_Plasma_Olink_Final.rds"
data.output.path.plasma <- file.path("../Data", file.name)
saveRDS(plasma_combined_renamed, data.output.path.plasma)

In [92]:
file.name <- "MM_BMIF_Olink_Final.rds"
data.output.path.bm <- file.path("../Data", file.name)
saveRDS(bm_renamed, data.output.path.bm)

In [93]:
unique_bm_visits <- distinct(bm_renamed[, c('subject.subjectGuid','SampleKitGuid','sample.visitDetails')])
unique_plasma_visits <- distinct(plasma_combined_renamed[cohort.cohortGuid=='FH1', c('subject.subjectGuid','sample.sampleKitGuid',
                                                                                     'sample.visitDetails','cohort.cohortGuid')])

In [94]:
write.csv(unique_plasma_visits, file='../../data/olink/fh1_unique_kits_plasma.csv')
write.csv(unique_bm_visits, file='../../data/olink/fh1_unique_kits_bm.csv')

In [96]:
sessionInfo()

R version 4.3.3 (2024-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /home/workspace/environment/olink/lib/libopenblasp-r0.3.27.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8    LC_NUMERIC=C        LC_TIME=C          
 [4] LC_COLLATE=C        LC_MONETARY=C       LC_MESSAGES=C      
 [7] LC_PAPER=C          LC_NAME=C           LC_ADDRESS=C       
[10] LC_TELEPHONE=C      LC_MEASUREMENT=C    LC_IDENTIFICATION=C

time zone: America/Los_Angeles
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] magrittr_2.0.3    lubridate_1.9.3   forcats_1.0.0     stringr_1.5.1    
 [5] purrr_1.1.0       readr_2.1.5       tidyr_1.3.1       tibble_3.3.0     
 [9] tidyverse_2.0.0   ggpubr_0.6.1      patchwork_1.2.0   dplyr_1.1.4      
[13] ggplot2_3.5.2     data.table_1.17.8 hise_2.16.0      

loaded via a namespace 